# AGEF-Net, Part 2: CTU-13 robustness experiments and evidence packaging
Run this notebook in Google Colab (a high-RAM runtime is helpful). It rebuilds only the CTU-13 part, so it does not need the CICIDS2017 data or the large models of Part 1.

**Order:** install, mount Drive, load GeoLite2, rebuild CTU-13, helper functions (R2), then R3 (validation-fitted fusion), R4 (threshold-matched baselines), R5 (leave-one-scenario-out), M1 (repeated splits), M2 (sensitivity with all flows) and R7 (evidence package with provenance).

**Before running:** place `GeoLite2-City.mmdb` (free MaxMind account, not redistributed here) at the path set in the GeoLite2 cell.

In [ ]:
!pip install -q geoip2 lightgbm xgboost scikit-learn pandas numpy scipy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import geoip2.database, ipaddress
GEOLITE_DB_PATH = '/content/drive/MyDrive/CICIDS2017/GeoLite2-City.mmdb'   # <- change if your file is elsewhere
geo_reader = geoip2.database.Reader(GEOLITE_DB_PATH)
def get_country(ip):
    try:
        ip_obj = ipaddress.ip_address(ip)
        if ip_obj.is_private:
            return 'Not Geolocatable (Private/Internal IP)'
        response = geo_reader.city(ip)
        return response.country.name or 'Unknown'
    except Exception:
        return 'Not Geolocatable (Invalid IP)'
print(get_country('8.8.8.8')); print(get_country('192.168.1.5'))

## Rebuild CTU-13 data + base models (about 5-10 min incl. download)

In [ ]:
import os, gc, subprocess, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

SEED = 42
np.random.seed(SEED)
DATA_DIR = '/content'          # CTU-13 files are downloaded here (re-downloaded if missing)

scenarios = {
    42: "https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-42/capture20110810.binetflow.2format",
    43: "https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-43/capture20110811.binetflow.2format",
    44: "https://mcfp.felk.cvut.cz/publicDatasets/CTU-Malware-Capture-Botnet-44/capture20110812.binetflow.2format",
}
cols = ['saddr','daddr','proto','sport','dport','state','stos','dtos','swin','dwin','shops','dhops',
        'stime','ltime','sttl','dttl','tcprtt','synack','ackdat','spkts','dpkts','sbytes','dbytes',
        'sappbytes','dappbytes','dur','pkts','bytes','appbytes','rate','srate','drate','label']
keep_cols = ['saddr','daddr','proto','state','sttl','dttl','spkts','dpkts','sbytes','dbytes',
             'dur','rate','srate','drate','label']

# ---- 1) load (same 30% chunk sampling and seed as the original cell) ----
all_dfs = []
for sid, url in scenarios.items():
    fname = f'{DATA_DIR}/ctu13_scenario{sid}.binetflow'
    if not os.path.exists(fname):
        print(f'Downloading scenario {sid} ...')
        subprocess.run(['wget', '-q', '-O', fname, url])
    parts = []
    for chunk in pd.read_csv(fname, names=cols, usecols=keep_cols, low_memory=False, chunksize=200000):
        chunk = chunk[chunk['saddr'] != 'SrcAddr']
        parts.append(chunk.sample(frac=0.3, random_state=42) if len(chunk) > 50000 else chunk)
    d = pd.concat(parts, ignore_index=True); d['scenario'] = sid
    all_dfs.append(d); print(f'  scenario {sid}: {len(d)} rows kept'); del parts; gc.collect()
df_multi = pd.concat(all_dfs, ignore_index=True); del all_dfs; gc.collect()
df_multi['Label'] = df_multi['label'].astype(str).str.lower().apply(lambda x: 1 if 'botnet' in x else 0)

# ---- 2) features (identical to the original cell) ----
feature_cols = ['dur','spkts','dpkts','sbytes','dbytes','rate','srate','drate','sttl','dttl']
df_model = df_multi.copy()
for c in feature_cols:
    df_model[c] = pd.to_numeric(df_model[c], errors='coerce')
df_model = df_model.dropna(subset=feature_cols)

top_states = df_model['state'].value_counts().head(10).index
df_model['state_grouped'] = df_model['state'].where(df_model['state'].isin(top_states), 'Other')
top_protos = df_model['proto'].value_counts().head(5).index
df_model['proto_grouped'] = df_model['proto'].where(df_model['proto'].isin(top_protos), 'Other')
df_model = pd.concat([df_model, pd.get_dummies(df_model['proto_grouped'], prefix='proto'),
                      pd.get_dummies(df_model['state_grouped'], prefix='state')], axis=1)
feature_cols_final = feature_cols + [c for c in df_model.columns
    if (c.startswith('proto_') or c.startswith('state_')) and c not in ['proto_grouped', 'state_grouped']]

# ---- 3) data-attrition audit (this replaces cell R1) ----
audit = pd.DataFrame({
    'rows_loaded': df_multi.groupby('scenario').size(),
    'attacks_loaded': df_multi.groupby('scenario')['Label'].sum(),
    'rows_modelled': df_model.groupby('scenario').size(),
    'attacks_modelled': df_model.groupby('scenario')['Label'].sum()}).fillna(0).astype(int)
audit['attacks_dropped'] = audit['attacks_loaded'] - audit['attacks_modelled']
audit['pct_attacks_dropped'] = (100 * audit['attacks_dropped'] / audit['attacks_loaded']).round(1)
print('\n=== DATA ATTRITION AUDIT ==='); print(audit)
audit.to_csv('table_data_attrition_audit.csv')
tmp = df_multi[feature_cols].apply(pd.to_numeric, errors='coerce')
lost = tmp[tmp.isna().any(axis=1) & (df_multi['Label'] == 1)]
print('\nMissing values in the dropped ATTACK rows, by column:'); print(lost.isna().sum())
del tmp, lost, df_multi; gc.collect()

# ---- 4) same stratified split as the paper, then free memory ----
X_ctu = df_model[feature_cols_final]; y_ctu = df_model['Label']
X_train_ctu, X_test_ctu, y_train_ctu, y_test_ctu = train_test_split(
    X_ctu, y_ctu, test_size=0.2, stratify=y_ctu, random_state=42)
df_model = df_model[['saddr', 'daddr', 'scenario', 'Label']]       # keep only what R2-R5 need
gc.collect()

# ---- 5) base models (LightGBM = D, XGBoost = C) ----
ctu_model = LGBMClassifier(n_estimators=200, random_state=SEED, n_jobs=-1, verbose=-1).fit(X_train_ctu, y_train_ctu)
xgb_ctu = XGBClassifier(random_state=SEED, n_jobs=-1, eval_metric='logloss', tree_method='hist').fit(X_train_ctu, y_train_ctu)

# sanity check: should be close to the paper's base model (F1 0.827, recall 0.775, precision ~0.887)
yp = ctu_model.predict(X_test_ctu); pr = ctu_model.predict_proba(X_test_ctu)[:, 1]
print('\nLightGBM base on the test split ->',
      'P=%.4f R=%.4f F1=%.4f AUC=%.4f' % (precision_score(y_test_ctu, yp), recall_score(y_test_ctu, yp),
                                            f1_score(y_test_ctu, yp), roc_auc_score(y_test_ctu, pr)))
print('train/test rows:', len(X_train_ctu), len(X_test_ctu), '| train attacks:', int(y_train_ctu.sum()))


## R2 helpers

In [ ]:
import ipaddress
from sklearn.metrics import (precision_score, recall_score, f1_score, roc_auc_score,
                             average_precision_score, precision_recall_curve)

def _valid_ip(ip):
    try:
        ipaddress.ip_address(str(ip)); return True
    except ValueError:
        return False

def geo_signals(frame):
    # G = 1 if source and destination countries differ AND both are geolocatable (Eq. 5)
    # E = 1 if both IP fields are present and valid IP addresses
    # Only unique IPs are looked up, so this is fast.
    uniq = pd.unique(pd.concat([frame['saddr'], frame['daddr']]))
    cmap = {ip: get_country(ip) for ip in uniq}
    vmap = {ip: _valid_ip(ip) for ip in uniq}
    s = frame['saddr'].map(cmap); d = frame['daddr'].map(cmap)
    ok = ~(s.str.startswith('Not Geolocatable') | d.str.startswith('Not Geolocatable') |
           (s == 'Unknown') | (d == 'Unknown'))
    G = ((s != d) & ok).astype(float).values
    E = (frame['saddr'].map(vmap) & frame['daddr'].map(vmap)).astype(float).values
    return G, E, s.values, d.values

def metrics(y, pred, score=None):
    out = {'Precision': precision_score(y, pred, zero_division=0),
           'Recall': recall_score(y, pred), 'F1': f1_score(y, pred)}
    if score is not None:
        out['ROC-AUC'] = roc_auc_score(y, score)
        out['PR-AUC'] = average_precision_score(y, score)
    return {k: round(float(v), 4) for k, v in out.items()}

def bootstrap_ci(y, pred, n_boot=500, seed=SEED):
    # seeded percentile bootstrap (Eq. 16) for Precision / Recall / F1
    rng = np.random.default_rng(seed)
    y = np.asarray(y).astype(bool); pred = np.asarray(pred).astype(bool); n = len(y)
    P, R, F = [], [], []
    for _ in range(n_boot):
        i = rng.integers(0, n, n); yt, yp = y[i], pred[i]
        tp = (yt & yp).sum(); fp = (~yt & yp).sum(); fn = (yt & ~yp).sum()
        p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn) if tp + fn else 0.0
        P.append(p); R.append(r); F.append(2*p*r/(p + r) if p + r else 0.0)
    q = lambda a: (round(float(np.percentile(a, 2.5)), 4), round(float(np.percentile(a, 97.5)), 4))
    return {'Precision': q(P), 'Recall': q(R), 'F1': q(F)}

def mcnemar_exact(y, pred1, pred2):
    from scipy.stats import binomtest
    y = np.asarray(y); c1 = (np.asarray(pred1) == y); c2 = (np.asarray(pred2) == y)
    b = int((c1 & ~c2).sum()); c = int((~c1 & c2).sum())
    return (1.0 if b + c == 0 else float(binomtest(min(b, c), b + c, 0.5).pvalue)), b, c

def best_f1_threshold(y, score):
    p, r, t = precision_recall_curve(y, score)
    f = 2 * p[:-1] * r[:-1] / np.clip(p[:-1] + r[:-1], 1e-12, None)
    return float(t[int(np.argmax(f))])
print('Helpers ready.')

## R3 fusion with validation-fitted weights/thresholds

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

# 1) split the TRAINING part again into fit (75%) and validation (25%)
Xa, Xv, ya, yv = train_test_split(X_train_ctu, y_train_ctu, test_size=0.25,
                                  stratify=y_train_ctu, random_state=SEED)
lgb_a = LGBMClassifier(n_estimators=200, random_state=SEED, n_jobs=-1).fit(Xa, ya)
xgb_a = XGBClassifier(n_estimators=200, random_state=SEED, n_jobs=-1,
                      eval_metric='logloss', tree_method='hist').fit(Xa, ya)

def get_signals(X):
    C = xgb_a.predict_proba(X)[:, 1]            # confidence  (XGBoost)
    D = lgb_a.predict_proba(X)[:, 1]            # domain model (LightGBM)
    G, E, _, _ = geo_signals(df_model.loc[X.index])
    return C, G, D, E

Cv, Gv, Dv, Ev = get_signals(Xv)
Ct, Gt, Dt, Et = get_signals(X_test_ctu)

# 2) learned weights on validation data
stk = LogisticRegression(max_iter=1000, random_state=SEED).fit(np.c_[Cv, Gv, Dv, Ev], yv)
print('Learned coefficients  C, G, D, E =', np.round(stk.coef_[0], 3), '| intercept', np.round(stk.intercept_, 3))
agef_learned_v = stk.predict_proba(np.c_[Cv, Gv, Dv, Ev])[:, 1]
agef_learned_t = stk.predict_proba(np.c_[Ct, Gt, Dt, Et])[:, 1]

# 3) fixed (proposed) weights, same retrained models
w = dict(C=0.40, G=0.25, D=0.20, E=0.15)
agef_fixed_v = w['C']*Cv + w['G']*Gv + w['D']*Dv + w['E']*Ev
agef_fixed_t = w['C']*Ct + w['G']*Gt + w['D']*Dt + w['E']*Et

# 4) thresholds tuned on VALIDATION only (best F1), then applied once to the test set
systems = {
    'LightGBM alone (D), thr=0.5':            (Dv, Dt, 0.5),
    'LightGBM alone (D), tuned thr':          (Dv, Dt, None),
    'XGBoost alone (C), thr=0.5':             (Cv, Ct, 0.5),
    'XGBoost alone (C), tuned thr':           (Cv, Ct, None),
    'AGEF-Net fixed weights, thr=0.5':        (agef_fixed_v, agef_fixed_t, 0.5),
    'AGEF-Net fixed weights, tuned thr':      (agef_fixed_v, agef_fixed_t, None),
    'AGEF-Net LEARNED weights, tuned thr':    (agef_learned_v, agef_learned_t, None),
}
rows, preds = [], {}
for name, (sv, st, thr) in systems.items():
    thr = best_f1_threshold(yv, sv) if thr is None else thr
    pred = (st >= thr).astype(int); preds[name] = pred
    rows.append({'System': name, 'thr': round(thr, 3), **metrics(y_test_ctu, pred, st)})
fusion_table = pd.DataFrame(rows)
print(fusion_table.to_string(index=False))
fusion_table.to_csv('table_fusion_comparison_validation_tuned.csv', index=False)

# 5) is the learned fusion different from the best single model? (exact McNemar + seeded CIs)
best_single = fusion_table[fusion_table['System'].str.contains('alone')].sort_values('F1').iloc[-1]['System']
p, b, c = mcnemar_exact(y_test_ctu, preds[best_single], preds['AGEF-Net LEARNED weights, tuned thr'])
print(f'\nMcNemar: {best_single}  vs  learned AGEF-Net -> p={p:.4g} (b={b}, c={c})')
print('Bootstrap 95% CI, learned AGEF-Net:', bootstrap_ci(y_test_ctu, preds['AGEF-Net LEARNED weights, tuned thr']))
print('Bootstrap 95% CI, ', best_single, ':', bootstrap_ci(y_test_ctu, preds[best_single]))

In [ ]:
import shutil, glob, os
os.makedirs('/content/drive/MyDrive/agef_backup', exist_ok=True)
for f in glob.glob('/content/table_*.csv') + glob.glob('/content/shap_*.png'):
    shutil.copy(f, '/content/drive/MyDrive/agef_backup/')
print('Saved to Drive:', sorted(os.listdir('/content/drive/MyDrive/agef_backup')))

## R4 threshold-matched baselines

In [ ]:
C0 = xgb_ctu.predict_proba(X_test_ctu)[:, 1]
D0 = ctu_model.predict_proba(X_test_ctu)[:, 1]
G0, E0, _, _ = geo_signals(df_model.loc[X_test_ctu.index])
AGEF0 = 0.40*C0 + 0.25*G0 + 0.20*D0 + 0.15*E0
pred_agef0 = (AGEF0 >= 0.5).astype(int)
target_recall = recall_score(y_test_ctu, pred_agef0)

def precision_at_recall(y, score, target):
    p, r, t = precision_recall_curve(y, score)
    ok = np.where(r[:-1] >= target)[0]
    i = ok[np.argmax(p[:-1][ok])]
    return round(float(p[i]), 4), round(float(r[i]), 4), round(float(t[i]), 4)

print(f'AGEF-Net (fixed weights, thr 0.5): {metrics(y_test_ctu, pred_agef0, AGEF0)}')
for nm, sc in [('LightGBM alone (D)', D0), ('XGBoost alone (C)', C0)]:
    p, r, t = precision_at_recall(y_test_ctu, sc, target_recall)
    print(f'{nm} forced to recall >= {target_recall:.4f}: precision={p}, recall={r}, threshold={t}, PR-AUC={average_precision_score(y_test_ctu, sc):.4f}, ROC-AUC={roc_auc_score(y_test_ctu, sc):.4f}')
print(f'AGEF-Net PR-AUC={average_precision_score(y_test_ctu, AGEF0):.4f}, ROC-AUC={roc_auc_score(y_test_ctu, AGEF0):.4f}')
print('\nIf a single model matches AGEF-Net recall with HIGHER precision, say so in the paper: the gain is a threshold effect, not fusion.')

## R5 leave-one-scenario-out

In [ ]:
rows = []
for held in sorted(df_model['scenario'].unique()):
    tr = df_model['scenario'] != held
    Xtr, ytr = X_ctu[tr.values], y_ctu[tr.values]
    Xte, yte = X_ctu[~tr.values], y_ctu[~tr.values]
    if yte.sum() == 0 or ytr.sum() == 0:
        continue
    m_l = LGBMClassifier(n_estimators=200, random_state=SEED, n_jobs=-1).fit(Xtr, ytr)
    m_x = XGBClassifier(n_estimators=200, random_state=SEED, n_jobs=-1, eval_metric='logloss', tree_method='hist').fit(Xtr, ytr)
    Cx, Dx = m_x.predict_proba(Xte)[:, 1], m_l.predict_proba(Xte)[:, 1]
    Gx, Ex, _, _ = geo_signals(df_model.loc[Xte.index])
    Ax = 0.40*Cx + 0.25*Gx + 0.20*Dx + 0.15*Ex
    for nm, sc in [('LightGBM alone', Dx), ('XGBoost alone', Cx), ('AGEF-Net (fixed weights)', Ax)]:
        rows.append({'Held-out scenario': held, 'System': nm, **metrics(yte, (sc >= 0.5).astype(int), sc)})
loso = pd.DataFrame(rows)
print(loso.to_string(index=False))
loso.to_csv('table_leave_one_scenario_out.csv', index=False)
print('\nNote: scenarios 42/43 are Neris, 44 is Rbot - holding out 44 tests a new botnet family.')

In [ ]:
import shutil, glob, os
os.makedirs('/content/drive/MyDrive/agef_backup', exist_ok=True)
for f in glob.glob('/content/table_*.csv') + glob.glob('/content/shap_*.png'):
    shutil.copy(f, '/content/drive/MyDrive/agef_backup/')
print('Saved to Drive:', sorted(os.listdir('/content/drive/MyDrive/agef_backup')))

## M1. Repeated stratified splits (mean ± SD)

In [ ]:
# ===== M1. Repeated-split evaluation (mean ± SD over several stratified splits) =====
import os, time, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_recall_curve, average_precision_score
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

SPLIT_SEEDS   = [42, 43, 44, 45, 46]   # 42 = the split used in the paper
LEARNED_SEEDS = [42, 43, 44]           # learned-weight fusion needs 2 extra model fits per seed; use [] to skip
OUT_CSV = '/content/drive/MyDrive/agef_backup/table_multisplit_raw.csv'
try:
    os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
except Exception:
    OUT_CSV = 'table_multisplit_raw.csv'

# geolocation signals for every flow, computed once (fast: only unique IPs are looked up)
_G, _E, _, _ = geo_signals(df_model)
G_all = pd.Series(_G, index=df_model.index); E_all = pd.Series(_E, index=df_model.index)
W = dict(C=0.40, G=0.25, D=0.20, E=0.15)

def precision_at_recall(y, score, target):
    p, r, t = precision_recall_curve(y, score)
    ok = np.where(r[:-1] >= target)[0]
    return float(p[ok[np.argmax(p[:-1][ok])]])

def row(seed, system, y, pred, score, **extra):
    m = metrics(y, pred, score)
    return {'split_seed': seed, 'System': system, **m, **extra}

rows = []
for seed in SPLIT_SEEDS:
    t0 = time.time()
    Xtr, Xte, ytr, yte = train_test_split(X_ctu, y_ctu, test_size=0.2, stratify=y_ctu, random_state=seed)
    lgb = LGBMClassifier(n_estimators=200, random_state=seed, n_jobs=-1, verbose=-1).fit(Xtr, ytr)
    xgb = XGBClassifier(random_state=seed, n_jobs=-1, eval_metric='logloss', tree_method='hist').fit(Xtr, ytr)
    D = lgb.predict_proba(Xte)[:, 1]; C = xgb.predict_proba(Xte)[:, 1]
    G = G_all.loc[Xte.index].values; E = E_all.loc[Xte.index].values
    A = W['C']*C + W['G']*G + W['D']*D + W['E']*E
    rec_A = recall_score(yte, (A >= 0.5).astype(int))
    rows.append(row(seed, 'LightGBM alone (thr 0.5)', yte, (D >= 0.5).astype(int), D))
    rows.append(row(seed, 'XGBoost alone (thr 0.5)',  yte, (C >= 0.5).astype(int), C))
    rows.append(row(seed, 'AGEF-Net fixed weights (thr 0.5)', yte, (A >= 0.5).astype(int), A))
    # precision of each single model when forced to AGEF-Net's recall
    rows.append({'split_seed': seed, 'System': 'LightGBM at AGEF recall', 'Precision': round(precision_at_recall(yte, D, rec_A), 4), 'Recall': round(rec_A, 4)})
    rows.append({'split_seed': seed, 'System': 'XGBoost at AGEF recall',  'Precision': round(precision_at_recall(yte, C, rec_A), 4), 'Recall': round(rec_A, 4)})

    if seed in LEARNED_SEEDS:      # validation-fitted weights + thresholds (as in R3), models fitted on 75% of the training part
        Xa, Xv, ya, yv = train_test_split(Xtr, ytr, test_size=0.25, stratify=ytr, random_state=seed)
        lg2 = LGBMClassifier(n_estimators=200, random_state=seed, n_jobs=-1, verbose=-1).fit(Xa, ya)
        xg2 = XGBClassifier(random_state=seed, n_jobs=-1, eval_metric='logloss', tree_method='hist').fit(Xa, ya)
        Cv, Dv = xg2.predict_proba(Xv)[:, 1], lg2.predict_proba(Xv)[:, 1]
        Ct, Dt = xg2.predict_proba(Xte)[:, 1], lg2.predict_proba(Xte)[:, 1]
        Gv, Ev = G_all.loc[Xv.index].values, E_all.loc[Xv.index].values
        stk = LogisticRegression(max_iter=1000).fit(np.c_[Cv, Gv, Dv, Ev], yv)
        Al_v = stk.predict_proba(np.c_[Cv, Gv, Dv, Ev])[:, 1]; Al_t = stk.predict_proba(np.c_[Ct, G, Dt, E])[:, 1]
        thr_l, thr_x = best_f1_threshold(yv, Al_v), best_f1_threshold(yv, Cv)
        rows.append(row(seed, 'XGBoost alone (val-tuned thr)', yte, (Ct >= thr_x).astype(int), Ct))
        rows.append(row(seed, 'AGEF-Net learned weights (val-tuned thr)', yte, (Al_t >= thr_l).astype(int), Al_t,
                        w_C=round(stk.coef_[0][0], 3), w_G=round(stk.coef_[0][1], 3), w_D=round(stk.coef_[0][2], 3), w_E=round(stk.coef_[0][3], 3)))
        del lg2, xg2
    raw = pd.DataFrame(rows); raw.to_csv(OUT_CSV, index=False)          # saved after every split (crash-safe)
    print(f'split {seed} done in {time.time()-t0:.0f}s   (saved to {OUT_CSV})')
    del lgb, xgb; import gc; gc.collect()

# ---- summary: mean ± SD over splits ----
raw = pd.DataFrame(rows)
def ms(x): return f'{x.mean():.4f} ± {x.std(ddof=1):.4f}' if x.notna().sum() > 1 else (f'{x.mean():.4f}' if x.notna().any() else '')
summary = raw.groupby('System', sort=False)[['Precision', 'Recall', 'F1', 'PR-AUC']].agg(ms)
summary.insert(0, 'n_splits', raw.groupby('System', sort=False).size())
print('\n=== MEAN ± SD OVER SPLITS ==='); print(summary.to_string())
summary.to_csv('table_multisplit_summary.csv')
try: summary.to_csv(OUT_CSV.replace('raw', 'summary'))
except Exception: pass

# ---- paired comparison on the same splits: AGEF-Net fixed vs XGBoost alone ----
pv = raw.pivot_table(index='split_seed', columns='System', values='F1')
d = pv['AGEF-Net fixed weights (thr 0.5)'] - pv['XGBoost alone (thr 0.5)']
print(f'\nF1(AGEF fixed) - F1(XGBoost): mean {d.mean():+.4f}, SD {d.std(ddof=1):.4f}, AGEF better on {(d > 0).sum()}/{len(d)} splits')
d2 = pv['AGEF-Net fixed weights (thr 0.5)'] - pv['LightGBM alone (thr 0.5)']
print(f'F1(AGEF fixed) - F1(LightGBM): mean {d2.mean():+.4f}, SD {d2.std(ddof=1):.4f}, AGEF better on {(d2 > 0).sum()}/{len(d2)} splits')


## M2. Sensitivity analysis: keep flows with missing TTL values

In [ ]:
# ===== M2. Sensitivity: keep the flows that were dropped for missing sttl/dttl (impute 0 + missing flags) =====
import os, gc, subprocess, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

def _load_all():
    out = []
    for sid, url in scenarios.items():
        fname = f'{DATA_DIR}/ctu13_scenario{sid}.binetflow'
        if not os.path.exists(fname):
            subprocess.run(['wget', '-q', '-O', fname, url])
        parts = []
        for ch in pd.read_csv(fname, names=cols, usecols=keep_cols, low_memory=False, chunksize=200000):
            ch = ch[ch['saddr'] != 'SrcAddr']
            parts.append(ch.sample(frac=0.3, random_state=42) if len(ch) > 50000 else ch)
        d = pd.concat(parts, ignore_index=True); d['scenario'] = sid; out.append(d); del parts; gc.collect()
    return pd.concat(out, ignore_index=True)

df2 = _load_all()
df2['Label'] = df2['label'].astype(str).str.lower().apply(lambda x: 1 if 'botnet' in x else 0)
for c in feature_cols:
    df2[c] = pd.to_numeric(df2[c], errors='coerce')
df2['sttl_missing'] = df2['sttl'].isna().astype(int)
df2['dttl_missing'] = df2['dttl'].isna().astype(int)
print('Flows loaded:', len(df2), '| attack flows:', int(df2['Label'].sum()))
print('Attack flows with missing dttl:', int(((df2['dttl_missing'] == 1) & (df2['Label'] == 1)).sum()))
df2[feature_cols] = df2[feature_cols].fillna(0)

ts = df2['state'].value_counts().head(10).index; df2['state_g'] = df2['state'].where(df2['state'].isin(ts), 'Other')
tp = df2['proto'].value_counts().head(5).index;  df2['proto_g'] = df2['proto'].where(df2['proto'].isin(tp), 'Other')
df2 = pd.concat([df2, pd.get_dummies(df2['proto_g'], prefix='proto'), pd.get_dummies(df2['state_g'], prefix='state')], axis=1)
feats2 = feature_cols + ['sttl_missing', 'dttl_missing'] + [c for c in df2.columns if (c.startswith('proto_') or c.startswith('state_')) and c not in ('proto_g', 'state_g')]

X2, y2 = df2[feats2], df2['Label']
Xtr, Xte, ytr, yte = train_test_split(X2, y2, test_size=0.2, stratify=y2, random_state=42)
lgb2 = LGBMClassifier(n_estimators=200, random_state=42, n_jobs=-1, verbose=-1).fit(Xtr, ytr)
xgb2 = XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss', tree_method='hist').fit(Xtr, ytr)
D2, C2 = lgb2.predict_proba(Xte)[:, 1], xgb2.predict_proba(Xte)[:, 1]
G2, E2, _, _ = geo_signals(df2.loc[Xte.index])
A2 = 0.40*C2 + 0.25*G2 + 0.20*D2 + 0.15*E2

res = [{'System': n, **metrics(yte, (s >= 0.5).astype(int), s)} for n, s in
       [('LightGBM alone', D2), ('XGBoost alone', C2), ('AGEF-Net fixed weights', A2)]]
sens = pd.DataFrame(res)
print(f'\n=== SENSITIVITY (all flows kept) | test flows {len(yte)}, test attack flows {int(yte.sum())}, train attack flows {int(ytr.sum())} ===')
print(sens.to_string(index=False)); sens.to_csv('table_sensitivity_all_flows.csv', index=False)

# recall on attack flows WITH vs WITHOUT a destination response
miss = Xte['dttl_missing'].values == 1
sub = []
for n, s in [('LightGBM alone', D2), ('XGBoost alone', C2), ('AGEF-Net fixed weights', A2)]:
    p = (s >= 0.5).astype(int); yy = yte.values
    for lab, mask in [('attacks WITH dttl', (~miss) & (yy == 1)), ('attacks WITHOUT dttl (one-way flows)', miss & (yy == 1))]:
        sub.append({'System': n, 'Subset': lab, 'n_attack_flows': int(mask.sum()),
                    'Recall': round(float(p[mask].mean()), 4) if mask.sum() else np.nan})
subdf = pd.DataFrame(sub); print('\n=== RECALL BY ATTACK-FLOW TYPE ==='); print(subdf.to_string(index=False))
subdf.to_csv('table_sensitivity_recall_by_flowtype.csv', index=False)

import shutil
for f in ['table_sensitivity_all_flows.csv', 'table_sensitivity_recall_by_flowtype.csv', 'table_multisplit_summary.csv']:
    try: shutil.copy(f, '/content/drive/MyDrive/agef_backup/')
    except Exception: pass
del df2, X2, Xtr, Xte, lgb2, xgb2; gc.collect()


## R7. Evidence package with provenance block
Edit `CTU_FILES` / `CICIDS_FILES`, then uncomment the example call to write a report and its hash certificate in one run.

In [ ]:
import hashlib, json, os, glob, platform, sys
from datetime import datetime, timezone

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for blk in iter(lambda: f.read(chunk), b''):
            h.update(blk)
    return h.hexdigest()

# TODO: list your actual source files here
CTU_FILES    = [f'/content/ctu13_scenario{s}.binetflow' for s in (42, 43, 44)]
CICIDS_FILES = glob.glob('/content/drive/MyDrive/CICIDS2017/*.csv')

# One mapping for the whole package. NOTE: attack_probability / risk_score belong to ISO/IEC 27042
# (analysis & interpretation), as the paper says - the old cell used "Clause 6.5" of 27037 for them.
ISO_MAP = {
    'record_id': 'ISO/IEC 27037 - identification (unique identifier)',
    'source_ip / destination_ip': 'ISO/IEC 27037 - identification of evidence source',
    'source_country / destination_country / border_status': 'Jurisdictional metadata (authors\' mapping to 27037 identification)',
    'attack_probability / risk_score / risk_level': 'ISO/IEC 27042 - analysis and interpretation',
    'generated_at_utc': 'ISO/IEC 27037 - documentation',
}

def write_evidence_package(records, dataset_name, out_prefix, source_files, model_desc):
    ts = datetime.now(timezone.utc).isoformat()
    meta = geo_reader.metadata()
    report = {
        'dataset': dataset_name, 'generated_at_utc': ts, 'n_records': len(records),
        'source_data_sha256': {os.path.basename(p): sha256_file(p) for p in source_files if os.path.exists(p)},
        'geolocation_db': {'file': os.path.basename(GEOLITE_DB_PATH), 'type': meta.database_type,
                           'build_utc': datetime.fromtimestamp(meta.build_epoch, timezone.utc).isoformat()},
        'model': model_desc,
        'software': {'python': sys.version.split()[0], 'numpy': np.__version__, 'pandas': pd.__version__,
                     'lightgbm': __import__('lightgbm').__version__, 'xgboost': __import__('xgboost').__version__},
        'iso_mapping': ISO_MAP, 'records': records}
    rpath = f'{out_prefix}_report.json'
    with open(rpath, 'w') as f: json.dump(report, f, indent=2, default=str)
    cert = {'evidence_file': rpath, 'hash_algorithm': 'SHA-256', 'hash_value': sha256_file(rpath),
            'hashed_at_utc': datetime.now(timezone.utc).isoformat()}
    cpath = f'{out_prefix}_hash_certificate.json'
    with open(cpath, 'w') as f: json.dump(cert, f, indent=2)
    assert sha256_file(rpath) == cert['hash_value']          # verify immediately
    print('Wrote', rpath, '+', cpath, '\nSHA-256:', cert['hash_value'])
    return report, cert

# Example (CTU-13): uses the evidence_records list built earlier
# rep, cert = write_evidence_package(evidence_records, 'CTU-13 scenarios 42/43/44 (real public IPs)',
#                                    'ctu13_evidence', CTU_FILES, 'LightGBM + XGBoost, native CTU-13 features, seed 42')
print('Evidence-package function ready (uncomment the example call to generate and replace the old reports).')